# FrozenLake DQN with autoregressive reasoning and REINFORCE

Use extra inference compute on a fixed observed history: the **same Mouse embedding and backbone** generate vocabulary tokens after the value token. The value head reads the hidden state at the **consumed end-of-generation token**. Set `END_GENERATION_TOKEN_ID` to override the stop token, or leave it `None` to use the checkpoint’s generation defaults (including multiple stop IDs when configured). Only a vocabulary projection and output norm are added from the same checkpoint; the usual delayed DQN target copy remains.

Install Mouse Core with its `examples` extra and use a CUDA notebook kernel. Both uncached training and paged generation use **FlexAttention only**. PyTorch currently supports CPU FlexAttention inference but not backward. `kernels` is declared in the `kernels` project extra; `USE_KERNELS` optionally enables compatible Hugging Face decoder kernels. Use standard Python 3.15; the `all` installation extra includes `kernels`. Continuous batching recycles rows in Mouse's own paged cache. It does not use a second generator or Transformers' batching engine.

This is a small executable experiment, not a claim that a few updates solve FrozenLake. Random replay can contain few successful paths; inspect its reward coverage.


In [ ]:
from collections.abc import Callable
from pathlib import Path
import random

import gymnasium as gym
from typing import Any, Literal, cast

import numpy as np
import torch
from torch import nn
from transformers import AutoTokenizer

from mouse_core import polyak_update

from mouse_core.models import Backbone, BaseBackbone, Embedding, MultiEmbedding, MLP, Model, save_model
from mouse_core.models.autoregressive_reasoner import AutoregressiveReasoner, ReasoningTrace
from mouse_core.models.vocabulary_head import VocabularyHead
from mouse_core.objectives import DqnObjective, boundary_discount, reinforce_loss


PRETRAINED = "Qwen/Qwen3-0.6B"
SEED = 7
IS_SLIPPERY = False
MAX_EPISODE_STEPS = 16
REPLAY_EPISODES = 64
EPISODES_PER_UPDATE = 2
TRAIN_STEPS = 10
LEARNING_RATE = 1e-5
GAMMA = 0.99
TARGET_TAU = 0.01
MAX_GRAD_NORM = 1.0
REINFORCE_WEIGHT = 1.0
REASONING_TEMPERATURE = 1.0
MAX_NEW_TOKENS = 128
END_GENERATION_TOKEN_ID: int | None = None
MAX_ACTIVE_REQUESTS = 8
GENERATION_BACKEND: Literal["uncached", "continuous"] = "continuous"
USE_KERNELS = False
GRADIENT_CHECKPOINTING = True
EVAL_EPISODES = 3
OUTPUT = Path("/tmp/mouse-frozenlake-autoregressive-dqn")


## Shared model and policy-gradient update

For the recorded action, let `y` be the detached DQN backup. Train the final-token value with ordinary DQN loss. The generated sequence receives reward `R = -(Q_final(a) - y)^2`. A pre-reasoning baseline uses `b = -(Q_value(a) - y)^2` with the same target. The extra loss is `-stopgrad(R-b) * sum(log p(token))`, including the end token, weighted by DQN participation. The normal DQN loss and REINFORCE loss update the shared model together.

Generation runs without an autograd graph. Recompute each sampled trace to score tokens and train values. There is no explicit imagination objective. A missing end token raises before an update; it is not forced or treated as a completed trace. Increase the budget if the checkpoint fails to stop. Scratchpads are discarded between environment actions.


In [ ]:
from tensordict import TensorDict
from mouse_core.data import make_token_batch
class ReasoningDqn(Model):
    def __init__(self, *, embedding: MultiEmbedding, backbone: BaseBackbone,
                 value_head: nn.Module, token_head: nn.Module, tokenizer_name: str) -> None:
        super().__init__()
        # Register the supplied objects themselves. No generator model is loaded.
        self.embedding = embedding
        self.backbone = backbone
        self.value_head = value_head
        self.reasoner = AutoregressiveReasoner(token_head)
        self.tokenizer_name = tokenizer_name

    def forward(self, tokens: TensorDict, traces: list[ReasoningTrace], *,
                temperature: float, score_tokens: bool) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor | None]:
        out = self.reasoner(tokens, traces, embedding=self.embedding, backbone=self.backbone,
                            tokenizer_name=self.tokenizer_name, temperature=temperature, score_tokens=score_tokens)
        return self.value_head(out.final_hidden), self.value_head(out.value_hidden), out.log_prob_sums


def pack_prefixes(prompts: list[list[int]]) -> TensorDict:
    """Standalone already-rendered decision prefixes, one isolated stream each.

    A normal Mouse DataLoader TensorDict can instead be passed directly to the
    reasoner; it splits that batch at its marked value tokens, retaining features.
    """
    if not prompts or any(not prompt for prompt in prompts):
        raise ValueError("Every decision prefix must end at a value token")
    lengths = np.array([len(prompt) for prompt in prompts], dtype=np.int64)
    ids = np.concatenate([np.array(prompt, dtype=np.int64) for prompt in prompts])
    return make_token_batch(
        token_ids=ids, numeric_values=np.zeros(len(ids)), numeric_mask=np.zeros(len(ids), dtype=bool),
        sequence_ids=np.repeat(np.arange(len(prompts)), lengths),
        head_token_mask=np.isin(np.arange(len(ids)), lengths.cumsum() - 1), num_sequences=len(prompts),
        tokenizer_names=("text",),
    )


def train_step(
    online: ReasoningDqn, target: ReasoningDqn, optimizer: torch.optim.Optimizer, objective: DqnObjective, *,
    tokens: TensorDict, objective_data: TensorDict, group_id: torch.Tensor,
    backend: Literal["uncached", "continuous"], end_token_id: int | None, max_new_tokens: int,
    max_active_requests: int, reasoning_temperature: float, reinforce_weight: float,
    target_tau: float, max_grad_norm: float,
) -> dict[str, float]:
    if not 0 < target_tau <= 1 or not 0 < max_grad_norm < float("inf") or not 0 < reinforce_weight < float("inf"):
        raise ValueError("Invalid target_tau, max_grad_norm, or reinforce_weight")
    if objective.value is not None or objective.cql_weight != 0 or objective.gate is not None:
        raise ValueError("This example's token reward is the plain, one-step, untransformed squared DQN error")
    device = next(online.parameters()).device
    tokens = tokens.to(device)
    objective_data = objective_data.to(device)
    group_id = group_id.to(device)
    online.train()
    target.eval()
    optimizer.zero_grad(set_to_none=True)
    traces = online.reasoner.generate(
        tokens, embedding=online.embedding, backbone=online.backbone, tokenizer_name=online.tokenizer_name,
        backend=backend, max_new_tokens=max_new_tokens, end_token_id=end_token_id,
        max_active_requests=max_active_requests, do_sample=True, temperature=reasoning_temperature,
    )
    q, q_before, log_probs = online(tokens, traces, temperature=reasoning_temperature, score_tokens=True)
    assert log_probs is not None
    with torch.no_grad():
        delayed_q, _, _ = target(tokens, traces, temperature=reasoning_temperature, score_tokens=False)
    losses, metrics = objective(
        objective_data=objective_data, group_id=group_id,
        head_step_ids=objective_data["head_step_ids"], predictions=q,
        backup_value_predictions=delayed_q, backup_action_predictions=delayed_q,
    )
    td_target = cast(torch.Tensor, metrics["backup"]).detach()
    weights = cast(torch.Tensor, metrics["backup_weight"]).detach()
    step_of = objective_data["head_step_ids"]
    next_rows = (step_of + 1).clamp_max(len(group_id) - 1)
    actions = torch.where(weights > 0, objective_data["action"][next_rows], 0).long()
    selected = q.gather(1, actions[:, None]).squeeze(1)
    selected_before = q_before.gather(1, actions[:, None]).squeeze(1)
    rewards = -(selected - td_target).square().detach()
    baseline = -(selected_before - td_target).square().detach()
    token_loss = reinforce_loss(log_prob_sums=log_probs, rewards=rewards, baseline=baseline, weights=weights)
    loss = losses["action_value"] + reinforce_weight * token_loss
    loss.backward()
    norm = nn.utils.clip_grad_norm_(online.parameters(), max_grad_norm, error_if_nonfinite=True)
    optimizer.step()
    polyak_update(target=target, source=online, tau=target_tau)
    return {
        "dqn_loss": float(losses["action_value"].detach()), "reinforce_loss": float(token_loss.detach()),
        "mean_advantage": float(((rewards - baseline) * weights).sum() / weights.sum().clamp_min(1)),
        "grad_norm": float(norm),
        "mean_generated_tokens": sum(len(trace.generated_ids) for trace in traces) / len(traces),
    }


`objective_data["head_step_ids"][p]` identifies the replay step that owns prediction row `p`. The objective uses this mapping to align predictions with actions, rewards and episode endings. Terminal steps have labels but no prediction; steps with multiple readouts repeat their step ID. Counts are derived internally and are not stored in `objective_data`.


## Fixed FrozenLake replay

Collect actual Gymnasium FrozenLake transitions once. Prefixes contain only observations, actions and rewards available at that decision; they never include future transitions or the hidden map. Arrival rows hold the previous action and its reward, as required by Mouse's DQN objective. Terminal and time-limit rows have no value readout. This finite-horizon example gives both boundaries zero continuation value.


In [ ]:
from tensordict import TensorDict
def make_replay(*, seed: int, num_episodes: int, horizon: int,
                is_slippery: bool) -> list[list[dict[str, Any]]]:
    """Collect real FrozenLake transitions once, then keep this replay fixed."""
    env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=is_slippery,
                   max_episode_steps=horizon)
    rng = random.Random(seed)
    episodes = []
    try:
        for episode in range(num_episodes):
            observation, _ = env.reset(seed=seed + episode)
            steps = [dict(observation=int(observation), action=None,
                          reward=0.0, terminated=False, truncated=False)]
            while True:
                action = rng.randrange(int(env.action_space.n))
                observation, reward, terminated, truncated, _ = env.step(action)
                done = terminated or truncated
                steps.append(dict(observation=int(observation), action=action,
                                  reward=float(reward), terminated=bool(terminated), truncated=bool(truncated)))
                if done:
                    break
            episodes.append(steps)
    finally:
        env.close()
    return episodes


def history_line(step: dict[str, Any]) -> str:
    if step["action"] is None:
        return f"State {step['observation']}.\n"
    return (f"Action {step['action']}, reward {step['reward']:g}, "
            f"state {step['observation']}.\n")


def replay_batch(episodes: list[list[dict[str, Any]]], *, encode_prompt: Callable[[str], list[int]],
                 device: torch.device) -> tuple[TensorDict, dict[str, torch.Tensor], torch.Tensor]:
    prompts: list[list[int]] = []
    columns: dict[str, list[Any]] = {key: [] for key in (
        "action", "reward", "terminated", "truncated",
    )}
    groups = []
    prediction_steps = []
    for group, episode in enumerate(episodes):
        if len(episode) < 2 or not (episode[-1]["terminated"] or episode[-1]["truncated"]):
            raise ValueError("Replay must contain complete FrozenLake episodes")
        history = ""
        for step in episode:
            history += history_line(step)
            if not (step["terminated"] or step["truncated"]):
                prediction_steps.append(len(groups))
                prompts.append(encode_prompt(history))
            for key in columns:
                columns[key].append(float("nan") if key == "action" and step[key] is None else step[key])
            groups.append(group)
    tokens = pack_prefixes(prompts)
    data = TensorDict({key: torch.tensor(values, device=device) for key, values in columns.items()},
                      batch_size=[], device=device)
    data["head_step_ids"] = torch.tensor(prediction_steps, dtype=torch.long, device=device)
    data["step_sequence_ids"] = torch.tensor(groups, dtype=torch.long, device=device)
    return tokens, data, data["step_sequence_ids"]


In [ ]:
replay = make_replay(seed=SEED, num_episodes=REPLAY_EPISODES,
                     horizon=MAX_EPISODE_STEPS, is_slippery=IS_SLIPPERY)
print(f"Fixed replay: {len(replay)} episodes, "
      f"{sum(len(ep) - 1 for ep in replay)} transitions, "
      f"{sum(ep[-1]['reward'] > 0 for ep in replay)} goals reached.")


## Build the shared model


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("This notebook trains with FlexAttention and requires CUDA; CPU supports inference only.")
device = torch.device("cuda")
torch.manual_seed(SEED)
rng = random.Random(SEED)
tokenizer = AutoTokenizer.from_pretrained(PRETRAINED)

# Supply these same objects from your existing Mouse model when adapting this example.
embedding = MultiEmbedding(embeddings={"text": Embedding(pretrained=PRETRAINED)})
backbone = Backbone(pretrained=PRETRAINED, train_kernel="flex", decode_kernel="flex", dtype=torch.float32)
backbone.gradient_checkpointing = GRADIENT_CHECKPOINTING
value_head = MLP(in_features=backbone.hidden_dim, out_features=4, hidden_dim=backbone.hidden_dim,
                 num_layers=1, scale=0.1, propagate_gradient=1.0)
online = ReasoningDqn(embedding=embedding, backbone=backbone, value_head=value_head,
                      token_head=VocabularyHead.from_pretrained(PRETRAINED), tokenizer_name="text").to(device)
target = online.copy().requires_grad_(False).eval()
if USE_KERNELS:
    backbone.enable_kernels()
    cast(Backbone, target.backbone).enable_kernels()
optimizer = torch.optim.AdamW(online.parameters(), lr=LEARNING_RATE, weight_decay=0.0)
objective = DqnObjective(
    reward=None, value=None, gate=None, temperature=0.0,
    discount=boundary_discount(gamma_step=GAMMA, gamma_terminated=0.0, gamma_truncated=0.0),
)


def encode_prompt(history: str) -> list[int]:
    messages = [{"role": "user", "content": (
        "Estimate action values in FrozenLake: a 4 by 4 grid, with states numbered row-major 0 to 15. "
        "Actions: 0=left, 1=down, 2=right, 3=up. The map is hidden. "
        "The goal rewards 1; a hole ends the episode with reward 0. "
        f"The episode ends after at most {MAX_EPISODE_STEPS} steps. "
        f"Slippery transitions: {IS_SLIPPERY}. "
        "Think briefly about the observed history, then stop.\nHistory:\n" + history
    )}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    return cast(list[int], tokenizer.encode(text + "Value:\n", add_special_tokens=False))


## Train on the fixed replay with fresh sampled reasoning each update


In [ ]:
for step in range(TRAIN_STEPS):
    episodes = rng.sample(replay, EPISODES_PER_UPDATE)
    tokens, data, groups = replay_batch(episodes, encode_prompt=encode_prompt, device=device)
    metrics = train_step(
        online, target, optimizer, objective, tokens=tokens, objective_data=data, group_id=groups,
        backend=GENERATION_BACKEND, end_token_id=END_GENERATION_TOKEN_ID, max_new_tokens=MAX_NEW_TOKENS,
        max_active_requests=MAX_ACTIVE_REQUESTS, reasoning_temperature=REASONING_TEMPERATURE,
        reinforce_weight=REINFORCE_WEIGHT, target_tau=TARGET_TAU, max_grad_norm=MAX_GRAD_NORM,
    )
    print(f"step={step + 1} {metrics}")


## Evaluate future actions with frozen weights and fresh reasoning


In [ ]:
online.eval()
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=IS_SLIPPERY,
               max_episode_steps=MAX_EPISODE_STEPS)
returns = []
try:
    with torch.no_grad():
        for episode in range(EVAL_EPISODES):
            observation, _ = env.reset(seed=SEED + REPLAY_EPISODES + episode)
            history = f"State {observation}.\n"
            total_reward = 0.0
            while True:
                tokens = pack_prefixes([encode_prompt(history)])
                traces = online.reasoner.generate(
                    tokens, embedding=online.embedding, backbone=online.backbone, tokenizer_name="text",
                    backend=GENERATION_BACKEND, end_token_id=END_GENERATION_TOKEN_ID,
                    max_new_tokens=MAX_NEW_TOKENS, max_active_requests=MAX_ACTIVE_REQUESTS,
                    do_sample=True, temperature=REASONING_TEMPERATURE,
                )
                q, _, _ = online(tokens, traces, temperature=REASONING_TEMPERATURE, score_tokens=False)
                action = int(q.argmax(-1).item())
                observation, reward, terminated, truncated, _ = env.step(action)
                total_reward += float(reward)
                history += f"Action {action}, reward {reward:g}, state {observation}.\n"
                if terminated or truncated:
                    break
            returns.append(total_reward)
finally:
    env.close()
print(f"Evaluation returns: {returns}; success rate: {np.mean(returns):.1%}")


## Save a local Mouse checkpoint


In [ ]:
if OUTPUT.exists():
    raise FileExistsError(f"Choose a new OUTPUT directory: {OUTPUT}")
save_model(model=online, path=OUTPUT)
tokenizer.save_pretrained(OUTPUT / "tokenizer")
print(f"Saved locally to {OUTPUT}")
